In [1]:
import requests
from bs4 import BeautifulSoup

from transformers import pipeline

from kvpress import RandomHeadPress
from kvpress.cache_utils import DynamicHeadCache

In [2]:
# Load pipeline
device = "cuda:0"
ckpt = "meta-llama/Meta-Llama-3.1-8B-Instruct"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, torch_dtype="auto")

# Load data
url = "https://en.wikipedia.org/wiki/Nvidia"
content = requests.get(url).content
soup = BeautifulSoup(content, "html.parser")
context = "".join([p.text for p in soup.find_all("p")]) + "\n\n"

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [3]:
# Load press
compression_ratio = 0.1
press = RandomHeadPress(compression_ratio)
cache = DynamicHeadCache()

# Run the pipeline on a single question
question = "Complete this sentence: The Nvidia GeForce Partner Program was a ..."
true_answer = "marketing program designed to provide partnering companies with benefits such as public relations support, video game bundling, and marketing development funds."
pred_answer = pipe(context, question=question, press=press, cache=cache)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")

Question:   Complete this sentence: The Nvidia GeForce Partner Program was a ...
Answer:     marketing program designed to provide partnering companies with benefits such as public relations support, video game bundling, and marketing development funds.
Prediction: The Nvidia GeForce Partner Program was a marketing program designed to provide partnering companies with benefits such as public relations support, video game bundling, and marketing development funds.


In [4]:
# Look at the cache

layer_idx = 0
batch_indices, head_indices, seq_indices = cache.indices[layer_idx]
print(f"Number of pruned indices: {len(batch_indices)}")
print(f"Per head: {head_indices.bincount().tolist()}")

Number of pruned indices: 6960
Per head: [890, 857, 874, 833, 864, 877, 874, 891]
